# Publications HAL par laboratoire (2015–2025)

Interrogation de l'API HAL (`https://api.archives-ouvertes.fr/search/`) par `structId_i` pour chaque laboratoire.

In [ ]:
import requests
import pandas as pd
import time
from tqdm.notebook import tqdm

## Liste des laboratoires

In [ ]:
LABS = [
    ("CMP",           {244423}),
    ("CRAN",          {185180, 1001}),
    ("CREATIS",       {139739}),
    ("CRIL",          {90448, 1628}),
    ("CRISTAL",       {410272, 389110, 388977, 390300, 183073, 111636, 24885, 2546, 186929}),
    ("DI ENS",        {25027}),
    ("CROSSING (IRL)",{1063106}),
    ("ETIS",          {1003474, 1061575, 1087906, 1003348}),
    ("FILOFOCS (UMI)",{1006288, 1006289}),
    ("GIPSA-Lab",     {1043333, 1042376, 24470}),
    ("GREYC",         {150}),
    ("G-SCOP",        {1043137, 1041927, 74240}),
    ("HEUDIASYC",     {389870}),
    ("I3S",           {13009, 552896, 1079434}),
    ("ICUBE",         {217648, 1073080}),
    ("IDRIS",         {1823}),
    ("IPAL (IRL)",    {542003, 220880, 138926}),
    ("IRIF",          {1005016, 444497}),
    ("IRISA",         {491183, 491231, 490899, 491188, 1092618, 491177, 1092619, 490900,
                       419364, 419370, 105128, 2494, 25255, 419365, 419367, 491230,
                       419363, 419366, 491232, 419362, 1099404, 545024, 1099406,
                       1099401, 1099402, 1099435, 525233, 1088566, 1088569, 495900,
                       489780, 1092631, 1092630, 1092628, 1092632, 1092626, 1092625, 1092629}),
    ("IRIT",          {34499, 1082335}),
    ("ISIR",          {541937, 96164}),
    ("JFLI",          {542009, 229050}),
    ("L2S",           {1051117, 1289}),
    ("LAAS",          {459}),
    ("LABRI",         {3102}),
    ("LAB-STICC",     {486345, 491660, 199324, 81533, 1089048}),
    ("LAMIH",         {1067790, 1303}),
    ("LAMSADE",       {989}),
    ("LIG",           {1043301, 1041964, 24471}),
    ("LIGM",          {1001627, 3210}),
    ("LIMOS",         {1063677, 490706, 857}),
    ("LIP",           {35418}),
    ("LIP6",          {541703, 233, 1095103}),
    ("LIPN",          {1000994, 994, 1086916, 1056718}),
    ("LIRIS",         {2003, 1086665}),
    ("LIRMM",         {181, 1071941}),
    ("LIS",           {527033, 199402, 199394, 862, 178374}),
    ("LISN",          {1061259, 1041968, 247329, 2544, 1050003, 81750}),
    ("LIX",           {2071, 1041697, 1071530, 1070048}),
    ("LMF",           {1065710}),
    ("LORIA",         {206040, 466633}),
    ("LS2N",          {1088564, 473973, 95421, 1693, 21439}),
    ("LMF-LSV",       {1065710, 1042689, 2571}),
    ("MDLS",          {210816}),
    ("RELAX",         {1040410, 528907}),
    ("ROOT (IRL)",    {389700}),
    ("STMS",          {541779, 1374}),
    ("TIMA",          {1043043, 1044023, 640}),
    ("TIMC IMAG",     {1043049, 1070489, 1042061, 707, 574002, 555959, 1056575}),
    ("VERIMAG",       {1043148, 1041816, 194}),
]

## Fonction d'interrogation de l'API HAL

In [ ]:
HAL_API_URL = "https://api.archives-ouvertes.fr/search/"
YEAR_MIN    = 2015
YEAR_MAX    = 2025
BATCH_SIZE  = 1000   # maximum accepté par l'API HAL
SLEEP_SEC   = 0.3    # pause entre requêtes pour ne pas surcharger l'API


def fetch_lab_publications(lab_name: str, struct_ids: set,
                           year_min: int = YEAR_MIN,
                           year_max: int = YEAR_MAX) -> list[dict]:
    """
    Récupère toutes les publications HAL d'un laboratoire
    identifié par ses structId_i, entre year_min et year_max.
    Gère la pagination automatiquement.
    """
    struct_query = " OR ".join(str(sid) for sid in struct_ids)
    q  = f"structId_i:({struct_query})"
    fq = f"producedDateY_i:[{year_min} TO {year_max}]"

    records = []
    start   = 0

    while True:
        params = {
            "q":    q,
            "fq":   fq,
            "fl":   "halId_s,title_s,authFullName_s,doiId_s,producedDateY_i",
            "rows": BATCH_SIZE,
            "start": start,
            "wt":   "json",
        }

        resp = requests.get(HAL_API_URL, params=params, timeout=60)
        resp.raise_for_status()
        data = resp.json()

        docs      = data["response"]["docs"]
        num_found = data["response"]["numFound"]

        for doc in docs:
            title_field = doc.get("title_s")
            records.append({
                "laboratoire": lab_name,
                "auteurs":     "; ".join(doc.get("authFullName_s") or []),
                "titre":       title_field[0] if title_field else None,
                "doi":         doc.get("doiId_s"),          # str ou None
                "annee":       doc.get("producedDateY_i"),
                "hal_id":      doc.get("halId_s"),
            })

        start += len(docs)
        if start >= num_found or not docs:
            break

        time.sleep(SLEEP_SEC)

    return records

## Collecte des données pour tous les laboratoires

In [ ]:
all_records = []
errors      = []

for lab_name, struct_ids in tqdm(LABS, desc="Laboratoires"):
    try:
        records = fetch_lab_publications(lab_name, struct_ids)
        all_records.extend(records)
        print(f"  {lab_name:20s} → {len(records):>5} publications")
    except Exception as e:
        errors.append((lab_name, str(e)))
        print(f"  ⚠ Erreur pour {lab_name}: {e}")

print(f"\nTotal brut : {len(all_records)} enregistrements")
if errors:
    print(f"Laboratoires en erreur : {[e[0] for e in errors]}")

## Construction du dataframe global

In [ ]:
df = pd.DataFrame(all_records, columns=["laboratoire", "auteurs", "titre", "doi", "annee", "hal_id"])

# Conversion de l'année en entier (nullable)
df["annee"] = pd.to_numeric(df["annee"], errors="coerce").astype("Int64")

print(f"Shape : {df.shape}")
print(f"Années : {sorted(df['annee'].dropna().unique().tolist())}")
print(f"Laboratoires : {df['laboratoire'].nunique()}")
df.head(10)

## Dédoublonnage (même hal_id dans plusieurs labos)

Une publication peut apparaître dans plusieurs laboratoires si elle porte plusieurs `structId_i`. On conserve toutes les lignes (un enregistrement par couple publication × laboratoire) mais on peut aussi créer une vue dédoublonnée.

In [ ]:
# Publications uniques (toutes affiliations confondues)
df_unique = df.drop_duplicates(subset=["hal_id"]).reset_index(drop=True)
print(f"Publications uniques (tous labos) : {len(df_unique)}")

# Nombre de publications par laboratoire
df.groupby("laboratoire").size().sort_values(ascending=False).rename("n_publications")

## Export CSV

In [ ]:
OUTPUT_ALL    = "hal_publications_par_labo.csv"
OUTPUT_UNIQUE = "hal_publications_uniques.csv"

df.to_csv(OUTPUT_ALL, index=False, encoding="utf-8-sig")
df_unique.to_csv(OUTPUT_UNIQUE, index=False, encoding="utf-8-sig")

print(f"Fichiers exportés :\n  {OUTPUT_ALL}\n  {OUTPUT_UNIQUE}")

---
# Croisement avec OpenAlex

Stratégie en deux passes sur `df_unique` :
1. **Passe DOI** : requête par lot (50 DOI à la fois) via `filter=doi:…`
2. **Passe titre** : pour les publications sans DOI (ou non trouvées par DOI), recherche sur `display_name.search` avec vérification de similarité (seuil 0.85)

> **Note** : la recherche par titre est heuristique — des faux positifs sont possibles sur des titres courts ou très génériques.

In [ ]:
import difflib
import unicodedata
import re

# ── Paramètres OpenAlex ──────────────────────────────────────────────────────
OPENALEX_URL    = "https://api.openalex.org/works"
OPENALEX_MAILTO = "votre.email@example.com"   # remplacer pour le « polite pool »
OA_BATCH_SIZE   = 50     # max DOI par requête (recommandé par OpenAlex)
OA_SLEEP_SEC    = 0.1    # pause entre requêtes
TITLE_SIM_MIN   = 0.85   # seuil de similarité pour la correspondance par titre


def _norm_title(title: str) -> str:
    """Normalise un titre pour la comparaison : minuscules, sans accents ni ponctuations."""
    if not title:
        return ""
    t = unicodedata.normalize("NFD", title.lower())
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")  # retire diacritiques
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def _norm_doi(doi: str) -> str:
    """Renvoie le DOI sans le préfixe URI, en minuscules."""
    if not doi:
        return ""
    doi = doi.strip().lower()
    for prefix in ("https://doi.org/", "http://doi.org/", "doi:"):
        if doi.startswith(prefix):
            doi = doi[len(prefix):]
    return doi


def _oa_params(**extra) -> dict:
    return {"mailto": OPENALEX_MAILTO, "select": "id,doi,display_name", **extra}


# ── Passe 1 : recherche par DOI (par lots) ───────────────────────────────────
def lookup_dois(doi_list: list[str]) -> dict[str, dict]:
    """
    Reçoit une liste de DOIs normalisés, renvoie un dict
    {doi_normalisé -> {"openalex_id": ..., "openalex_title": ...}}.
    """
    results = {}
    for i in range(0, len(doi_list), OA_BATCH_SIZE):
        batch = doi_list[i : i + OA_BATCH_SIZE]
        filter_val = "|".join(batch)
        params = _oa_params(filter=f"doi:{filter_val}", per_page=OA_BATCH_SIZE)
        try:
            resp = requests.get(OPENALEX_URL, params=params, timeout=30)
            resp.raise_for_status()
            for work in resp.json().get("results", []):
                raw_doi = work.get("doi") or ""
                nd = _norm_doi(raw_doi)
                if nd:
                    results[nd] = {
                        "openalex_id":    work.get("id"),
                        "openalex_title": work.get("display_name"),
                    }
        except Exception as e:
            print(f"  ⚠ Erreur batch DOI [{i}:{i+OA_BATCH_SIZE}] : {e}")
        time.sleep(OA_SLEEP_SEC)
    return results


# ── Passe 2 : recherche par titre (unitaire) ─────────────────────────────────
def lookup_title(title: str) -> dict | None:
    """
    Cherche un titre dans OpenAlex. Renvoie le premier résultat si la
    similarité normalisée dépasse TITLE_SIM_MIN, sinon None.
    """
    if not title or len(title.strip()) < 10:
        return None
    # Tronque à 200 caractères pour éviter des URLs trop longues
    search_title = title[:200]
    params = _oa_params(**{"filter": f'display_name.search:"{search_title}"', "per_page": 1})
    try:
        resp = requests.get(OPENALEX_URL, params=params, timeout=30)
        resp.raise_for_status()
        results = resp.json().get("results", [])
        if not results:
            return None
        work = results[0]
        oa_title = work.get("display_name") or ""
        sim = difflib.SequenceMatcher(
            None, _norm_title(title), _norm_title(oa_title)
        ).ratio()
        if sim >= TITLE_SIM_MIN:
            return {
                "openalex_id":    work.get("id"),
                "openalex_title": oa_title,
                "title_sim":      round(sim, 3),
            }
    except Exception as e:
        print(f"  ⚠ Erreur titre : {e}")
    return None

In [ ]:
# ── Initialisation des colonnes résultats ────────────────────────────────────
df_unique = df_unique.copy()
df_unique["in_openalex"]    = False
df_unique["match_method"]   = pd.NA        # "doi" | "titre"
df_unique["openalex_id"]    = pd.NA
df_unique["openalex_title"] = pd.NA
df_unique["title_sim"]      = pd.NA

# ── Passe 1 : DOI ────────────────────────────────────────────────────────────
has_doi = df_unique["doi"].notna() & (df_unique["doi"].str.strip() != "")
dois_norm = df_unique.loc[has_doi, "doi"].apply(_norm_doi).tolist()
doi_index = df_unique.loc[has_doi].index   # index dans df_unique

print(f"Publications avec DOI : {has_doi.sum()} / {len(df_unique)}")
print("Recherche par DOI dans OpenAlex…")

doi_results = lookup_dois(dois_norm)
print(f"  → {len(doi_results)} DOI trouvés dans OpenAlex")

# Appliquer les résultats DOI
for idx, doi_raw in zip(doi_index, dois_norm):
    if doi_raw in doi_results:
        hit = doi_results[doi_raw]
        df_unique.at[idx, "in_openalex"]    = True
        df_unique.at[idx, "match_method"]   = "doi"
        df_unique.at[idx, "openalex_id"]    = hit["openalex_id"]
        df_unique.at[idx, "openalex_title"] = hit["openalex_title"]

print(f"Publications trouvées via DOI : {(df_unique['match_method'] == 'doi').sum()}")

In [ ]:
# ── Passe 2 : titre (sans DOI OU non trouvé par DOI) ─────────────────────────
to_search_by_title = df_unique[
    (~df_unique["in_openalex"]) & (df_unique["titre"].notna())
].index

print(f"Publications à rechercher par titre : {len(to_search_by_title)}")
print("Recherche par titre dans OpenAlex (peut être long)…")

for idx in tqdm(to_search_by_title, desc="Recherche titre"):
    titre = df_unique.at[idx, "titre"]
    hit = lookup_title(titre)
    if hit:
        df_unique.at[idx, "in_openalex"]    = True
        df_unique.at[idx, "match_method"]   = "titre"
        df_unique.at[idx, "openalex_id"]    = hit["openalex_id"]
        df_unique.at[idx, "openalex_title"] = hit["openalex_title"]
        df_unique.at[idx, "title_sim"]      = hit["title_sim"]
    time.sleep(OA_SLEEP_SEC)

print(f"Publications trouvées via titre : {(df_unique['match_method'] == 'titre').sum()}")
print(f"Publications non trouvées : {(~df_unique['in_openalex']).sum()}")

## Taux de couverture global (df_unique)

In [ ]:
total   = len(df_unique)
found   = df_unique["in_openalex"].sum()
by_doi  = (df_unique["match_method"] == "doi").sum()
by_titl = (df_unique["match_method"] == "titre").sum()

print("=" * 50)
print(f"Publications HAL uniques           : {total:>7}")
print(f"Présentes dans OpenAlex            : {found:>7}  ({100*found/total:.1f}%)")
print(f"  dont trouvées par DOI            : {by_doi:>7}  ({100*by_doi/total:.1f}%)")
print(f"  dont trouvées par titre          : {by_titl:>7}  ({100*by_titl/total:.1f}%)")
print(f"Non trouvées dans OpenAlex         : {total-found:>7}  ({100*(total-found)/total:.1f}%)")
print("=" * 50)

# Détail par méthode de correspondance
df_unique.groupby("match_method", dropna=False).size().rename("count").to_frame()

## Taux de couverture par laboratoire (df)

On propage les résultats OpenAlex vers `df` (non dédoublonné) via `hal_id`.

In [ ]:
# Jointure df ← df_unique sur hal_id
oa_cols = ["hal_id", "in_openalex", "match_method", "openalex_id"]
df = df.drop(columns=[c for c in oa_cols[1:] if c in df.columns], errors="ignore")
df = df.merge(df_unique[oa_cols], on="hal_id", how="left")

# Taux par laboratoire
stats_labo = (
    df.groupby("laboratoire")
    .agg(
        n_hal        =("hal_id",      "count"),
        n_openalex   =("in_openalex", "sum"),
        n_doi        =("match_method", lambda x: (x == "doi").sum()),
        n_titre      =("match_method", lambda x: (x == "titre").sum()),
    )
    .assign(
        taux_openalex=lambda d: (d["n_openalex"] / d["n_hal"] * 100).round(1)
    )
    .sort_values("taux_openalex", ascending=False)
)

stats_labo

## Export final

In [ ]:
df_unique.to_csv("hal_openalex_uniques.csv",   index=False, encoding="utf-8-sig")
df.to_csv(       "hal_openalex_par_labo.csv",  index=False, encoding="utf-8-sig")
stats_labo.to_csv("couverture_openalex_par_labo.csv",        encoding="utf-8-sig")

print("Fichiers exportés :")
print("  hal_openalex_uniques.csv          (df_unique + colonnes OpenAlex)")
print("  hal_openalex_par_labo.csv         (df non dédoublonné + colonnes OpenAlex)")
print("  couverture_openalex_par_labo.csv  (stats par laboratoire)")